In [14]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import BertTokenizer

PROJECT_ROOT = "/Users/denis/Desktop/coursework/Course_work_community_detection"

DATA_PATH = os.path.join(
    PROJECT_ROOT,
    "data/raw/dataset-electronics-5M.parquet"
)

CHECKPOINT_PATH = os.path.join(
    PROJECT_ROOT,
    "models/epoch_64_encoder.pth"
)

TOKENIZER_DIR = os.path.join(
    PROJECT_ROOT,
    "models/rubert-tiny2-tokenizer"
)

DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

RANDOM_STATE = 42

print("Device:", DEVICE)
print("Checkpoint:", CHECKPOINT_PATH)
print("Tokenizer:", TOKENIZER_DIR)

Device: mps
Checkpoint: /Users/denis/Desktop/coursework/Course_work_community_detection/models/epoch_64_encoder.pth
Tokenizer: /Users/denis/Desktop/coursework/Course_work_community_detection/models/rubert-tiny2-tokenizer


In [15]:
tokenizer = BertTokenizer.from_pretrained(
    TOKENIZER_DIR,
    local_files_only=True
)

print("Tokenizer size:", len(tokenizer))

text = "Смартфон Apple iPhone 15 128GB"

print("Tokens:")
print(tokenizer.tokenize(text))

Tokenizer size: 83828
Tokens:
['Смарт', '##фон', 'Apple', 'iPhone', '15', '128', '##GB']


In [16]:
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint loaded")
print("Keys:", checkpoint.keys())
print("Meta:", checkpoint["meta"])

Checkpoint loaded
Keys: dict_keys(['meta', 'state_dict', 'optimizer', 'scheduler'])
Meta: {'epoch': 64, 'iter': 6400, 'time': 'Mon Sep  8 11:22:19 2025'}


In [17]:
print(
    "Parameters:",
    sum(p.numel() for p in encoder.parameters())
)

Parameters: 29096112


In [18]:
class BertLayer(nn.Module):
    def __init__(
        self,
        hidden_size=312,
        num_heads=12,
        intermediate_size=600
    ):
        super().__init__()

        self.attention = BertAttention(
            hidden_size,
            num_heads
        )

        self.intermediate = nn.Module()
        self.intermediate.dense = nn.Linear(
            hidden_size,
            intermediate_size
        )

        self.output = BertOutput(
            intermediate_size,
            hidden_size
        )

    def forward(self, x, attention_mask):
        x = self.attention(x, attention_mask)

        intermediate = F.gelu(
            self.intermediate.dense(x)
        )

        x = self.output(
            intermediate,
            x
        )

        return x

class BertAttention(nn.Module):
    def __init__(
        self,
        hidden_size,
        num_heads
    ):
        super().__init__()

        self.self = BertSelfAttention(
            hidden_size,
            num_heads
        )

        self.output = BertSelfOutput(
            hidden_size
        )

    def forward(self, x, attention_mask):
        attention_output = self.self(
            x,
            attention_mask
        )

        return self.output(
            attention_output,
            x
        )


class BertSelfAttention(nn.Module):
    def __init__(
        self,
        hidden_size,
        num_heads
    ):
        super().__init__()

        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads

        self.query = nn.Linear(
            hidden_size,
            hidden_size
        )

        self.key = nn.Linear(
            hidden_size,
            hidden_size
        )

        self.value = nn.Linear(
            hidden_size,
            hidden_size
        )

    def forward(self, x, attention_mask):
        batch_size, seq_len, hidden_size = x.shape

        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        q = q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        scores = torch.matmul(
            q,
            k.transpose(-1, -2)
        )

        scores = scores / np.sqrt(self.head_dim)

        scores = scores + attention_mask

        attention_probs = torch.softmax(
            scores,
            dim=-1
        )

        context = torch.matmul(
            attention_probs,
            v
        )

        context = context.transpose(
            1,
            2
        ).contiguous()

        context = context.view(
            batch_size,
            seq_len,
            hidden_size
        )

        return context


class BertSelfOutput(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()

        self.dense = nn.Linear(
            hidden_size,
            hidden_size
        )

        self.LayerNorm = nn.LayerNorm(
            hidden_size,
            eps=1e-12
        )

    def forward(self, x, residual):
        x = self.dense(x)

        return self.LayerNorm(
            x + residual
        )


class BertOutput(nn.Module):
    def __init__(
        self,
        intermediate_size,
        hidden_size
    ):
        super().__init__()

        self.dense = nn.Linear(
            intermediate_size,
            hidden_size
        )

        self.LayerNorm = nn.LayerNorm(
            hidden_size,
            eps=1e-12
        )

    def forward(self, x, residual):
        x = self.dense(x)

        return self.LayerNorm(
            x + residual
        )


class BertEncoder(nn.Module):
    def __init__(
        self,
        vocab_size=83828,
        hidden_size=312,
        num_layers=3,
        num_heads=12,
        intermediate_size=600,
        max_position_embeddings=2048
    ):
        super().__init__()

        self.embeddings = nn.Module()

        self.embeddings.word_embeddings = nn.Embedding(
            vocab_size,
            hidden_size
        )

        self.embeddings.position_embeddings = nn.Embedding(
            max_position_embeddings,
            hidden_size
        )

        self.embeddings.token_type_embeddings = nn.Embedding(
            2,
            hidden_size
        )

        self.embeddings.LayerNorm = nn.LayerNorm(
            hidden_size,
            eps=1e-12
        )

        self.encoder = nn.Module()

        self.encoder.layer = nn.ModuleList([
            BertLayer(
                hidden_size,
                num_heads,
                intermediate_size
            )
            for _ in range(num_layers)
        ])

    def forward(
        self,
        input_ids,
        attention_mask
    ):
        batch_size, seq_len = input_ids.shape

        position_ids = torch.arange(
            seq_len,
            device=input_ids.device
        ).unsqueeze(0).expand(
            batch_size,
            -1
        )

        token_type_ids = torch.zeros_like(
            input_ids
        )

        x = (
            self.embeddings.word_embeddings(input_ids)
            + self.embeddings.position_embeddings(position_ids)
            + self.embeddings.token_type_embeddings(token_type_ids)
        )

        x = self.embeddings.LayerNorm(x)

        extended_mask = attention_mask[
            :, None, None, :
        ].float()

        extended_mask = (
            1.0 - extended_mask
        ) * -10000.0

        for layer in self.encoder.layer:
            x = layer(
                x,
                extended_mask
            )

        return x

In [19]:
encoder = BertEncoder()

state_dict = checkpoint["state_dict"]

missing, unexpected = encoder.load_state_dict(
    state_dict,
    strict=False
)

print("Missing:", missing)
print("Unexpected:", unexpected)

print(
    "Parameters:",
    sum(p.numel() for p in encoder.parameters())
)

encoder = encoder.to(DEVICE)
encoder.eval()

Missing: []
Unexpected: []
Parameters: 29096112


BertEncoder(
  (embeddings): Module(
    (word_embeddings): Embedding(83828, 312)
    (position_embeddings): Embedding(2048, 312)
    (token_type_embeddings): Embedding(2, 312)
    (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True, bias=True)
  )
  (encoder): Module(
    (layer): ModuleList(
      (0-2): 3 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=312, out_features=312, bias=True)
            (key): Linear(in_features=312, out_features=312, bias=True)
            (value): Linear(in_features=312, out_features=312, bias=True)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=312, out_features=312, bias=True)
            (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True, bias=True)
          )
        )
        (intermediate): Module(
          (dense): Linear(in_features=312, out_features=600, bias=True)
        )
        (output): Bert

In [21]:
encoder = encoder.to(DEVICE)
encoder.eval()

text = "Смартфон Apple iPhone 15 128GB"

encoded = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=128
)

input_ids = encoded["input_ids"].to(DEVICE)
attention_mask = encoded["attention_mask"].to(DEVICE)

with torch.no_grad():
    output = encoder(
        input_ids,
        attention_mask
    )

print("Input:", input_ids.shape)
print("Output:", output.shape)

embedding = output[:, 0, :]

print("Embedding:", embedding.shape)
print("Norm:", torch.norm(embedding).item())

Input: torch.Size([1, 9])
Output: torch.Size([1, 9, 312])
Embedding: torch.Size([1, 312])
Norm: 16.20451545715332


In [24]:
import pyarrow.parquet as pq

parquet_file = pq.ParquetFile(DATA_PATH)

print("Количество строк:", parquet_file.metadata.num_rows)

Количество строк: 4793821


In [25]:
texts_test = []

for batch in parquet_file.iter_batches(
    batch_size=100,
    columns=["model_text"]
):
    batch_texts = batch.column("model_text").to_pylist()

    for text in batch_texts:
        if isinstance(text, bytes):
            text = text.decode("utf-8", errors="replace")

        texts_test.append(text)

        if len(texts_test) >= 5:
            break

    if len(texts_test) >= 5:
        break

for i, text in enumerate(texts_test):
    print(f"{i}: {text[:200]}")

0: Дисплей для OPPO Reno 13 F 4G In-Cell ЧерныйНе определенДисплей для OPPO Reno 13 F 4G In-Cell Черный идеально подойдет для замены Вашего разбитого дисплея. Однако, обратите внимание на важные моменты:
1: Дисплей для Tecno Spark 30C 4G ЧерныйНе определенДисплей для Tecno Spark 30C 4G Черный идеально подойдет для замены Вашего разбитого дисплея. Однако, обратите внимание на важные моменты: <br /> 1. Не 
2: Дисплей для OPPO Reno 12 F 4G/Realme 12 4G In-Cell (Premium Quality)Не определенДисплей для OPPO Reno 12 F 4G/Realme 12 4G In-Cell (Premium Quality) идеально подойдет для замены Вашего разбитого диспл
3: Дисплей для Xiaomi 13 (2211133C) OLEDНе определенДисплей для Xiaomi 13 (2211133C) OLED идеально подойдет для замены Вашего разбитого дисплея. Однако, обратите внимание на важные моменты: <br /> 1. Не 
4: Дисплей для Samsung Galaxy S23 в рамке Черный In-CellНе определенДисплей для Samsung Galaxy S23 в рамке Черный In-Cell идеально подойдет для замены Вашего разбитого дисплея. Однако,

In [26]:
def get_embeddings(
    texts,
    batch_size=8,
    max_length=512
):
    encoder.eval()

    embeddings = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        input_ids = encoded["input_ids"].to(DEVICE)
        attention_mask = encoded["attention_mask"].to(DEVICE)

        with torch.no_grad():
            output = encoder(
                input_ids,
                attention_mask
            )

        batch_embeddings = output[:, 0, :]

        embeddings.append(
            batch_embeddings.cpu().numpy()
        )

    return np.concatenate(embeddings, axis=0)

In [27]:
test_embeddings = get_embeddings(
    texts_test,
    batch_size=5,
    max_length=512
)

print("Shape:", test_embeddings.shape)
print("First embedding:", test_embeddings[0][:10])

Shape: (5, 312)
First embedding: [-1.2849951  -0.04631735 -1.1420527   0.32879043  0.24574357 -2.0508237
 -0.22076014 -0.31604913  0.02441844 -0.54522187]


In [29]:
embeddings_test_raw = []

for batch in parquet_file.iter_batches(
    batch_size=5,
    columns=["embedding"]
):
    embeddings_test_raw.extend(
        batch.column("embedding").to_pylist()
    )
    break

print("Количество:", len(embeddings_test_raw))
print("Размер первого:", len(embeddings_test_raw[0]))

Количество: 5
Размер первого: 3122


In [30]:
def decode_embedding(data):
    data = bytes(data)

    if len(data) != 3122:
        raise ValueError(
            f"Неожиданный размер embedding: {len(data)}"
        )

    values = np.empty(
        312,
        dtype=np.float64
    )

    for i in range(312):
        start = 2 + i * 10

        values[i] = np.frombuffer(
            data[start:start + 8],
            dtype="<f8"
        )[0]

    return values.astype(np.float32)


original_embeddings = np.stack([
    decode_embedding(x)
    for x in embeddings_test_raw
])

print(
    "Shape:",
    original_embeddings.shape
)

print(
    "First embedding:",
    original_embeddings[0][:10]
)

Shape: (5, 312)
First embedding: [-0.08203125 -0.00546265 -0.07910156  0.02307129  0.00726318 -0.15332031
 -0.02185059 -0.00970459  0.00848389 -0.04150391]


In [31]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = np.diag(
    cosine_similarity(
        test_embeddings,
        original_embeddings
    )
)

for i, similarity in enumerate(similarities):
    print(
        f"Text {i}: cosine similarity = {similarity:.6f}"
    )

Text 0: cosine similarity = 0.995285
Text 1: cosine similarity = 0.996131
Text 2: cosine similarity = 0.995412
Text 3: cosine similarity = 0.996360
Text 4: cosine similarity = 0.996685


Переходим к DeepCluster

In [32]:
SAMPLE_SIZE = 500_000
N_CLUSTERS = 10
MAX_LENGTH = 512
BATCH_SIZE = 16
LEARNING_RATE = 1e-5

print("Sample size:", SAMPLE_SIZE)
print("Clusters:", N_CLUSTERS)
print("Max length:", MAX_LENGTH)
print("Batch size:", BATCH_SIZE)

Sample size: 500000
Clusters: 10
Max length: 512
Batch size: 16


In [33]:
pf = pq.ParquetFile(DATA_PATH)

category_counts = {}

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=["category_id"]
):
    categories = batch.column("category_id").to_numpy()

    unique, counts = np.unique(
        categories,
        return_counts=True
    )

    for category, count in zip(unique, counts):
        category = int(category)
        category_counts[category] = (
            category_counts.get(category, 0) + int(count)
        )

print("Categories:", len(category_counts))
print("Total rows:", sum(category_counts.values()))

Categories: 122
Total rows: 4793821


In [34]:
rng = np.random.default_rng(RANDOM_STATE)

indices_by_category = {
    category: []
    for category in category_counts
}

row_start = 0

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=["category_id"]
):
    categories = batch.column("category_id").to_numpy()

    for local_idx, category in enumerate(categories):
        category = int(category)

        indices_by_category[category].append(
            row_start + local_idx
        )

    row_start += len(categories)

sample_indices = []

for category, indices in indices_by_category.items():
    sample_n = round(
        SAMPLE_SIZE *
        len(indices) /
        row_start
    )

    sample_n = min(
        sample_n,
        len(indices)
    )

    selected = rng.choice(
        indices,
        size=sample_n,
        replace=False
    )

    sample_indices.extend(selected)

sample_indices = np.array(
    sample_indices,
    dtype=np.int64
)

rng.shuffle(sample_indices)

if len(sample_indices) > SAMPLE_SIZE:
    sample_indices = rng.choice(
        sample_indices,
        size=SAMPLE_SIZE,
        replace=False
    )

print("Sample:", len(sample_indices))

Sample: 500000


In [35]:
from torch.utils.data import Dataset, DataLoader


class DeepClusterDataset(Dataset):
    def __init__(
        self,
        parquet_path,
        indices,
        pseudo_labels,
        tokenizer,
        max_length=512
    ):
        self.parquet_path = parquet_path
        self.indices = np.asarray(
            indices,
            dtype=np.int64
        )
        self.pseudo_labels = np.asarray(
            pseudo_labels,
            dtype=np.int64
        )
        self.tokenizer = tokenizer
        self.max_length = max_length

        self.pf = pq.ParquetFile(
            parquet_path
        )

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        row_idx = self.indices[idx]

        table = self.pf.read_row_group(
            self._find_row_group(row_idx),
            columns=["model_text"]
        )

        text = table["model_text"][
            row_idx - self._row_group_start
        ].as_py()

        if isinstance(text, bytes):
            text = text.decode(
                "utf-8",
                errors="replace"
            )

        encoded = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "label": torch.tensor(
                self.pseudo_labels[idx],
                dtype=torch.long
            )
        }

Создаём индекс row groups

In [36]:
pf = pq.ParquetFile(DATA_PATH)

row_group_starts = []
row_group_sizes = []

current_start = 0

for i in range(pf.num_row_groups):
    size = pf.metadata.row_group(i).num_rows

    row_group_starts.append(current_start)
    row_group_sizes.append(size)

    current_start += size

row_group_starts = np.array(row_group_starts)
row_group_sizes = np.array(row_group_sizes)

print("Row groups:", pf.num_row_groups)
print("Total rows:", current_start)

Row groups: 2108
Total rows: 4793821


In [37]:
def get_row_group(row_idx):
    group = np.searchsorted(
        row_group_starts,
        row_idx,
        side="right"
    ) - 1

    return group, row_idx - row_group_starts[group]

In [38]:
for idx in [0, 100, 100000, 1_000_000, 4_000_000]:
    group, local_idx = get_row_group(idx)
    print(
        idx,
        "→ row_group:",
        group,
        "local:",
        local_idx
    )

0 → row_group: 0 local: 0
100 → row_group: 0 local: 100
100000 → row_group: 38 local: 919
1000000 → row_group: 438 local: 487
4000000 → row_group: 1747 local: 243


In [39]:
class DeepClusterDataset(Dataset):
    def __init__(
        self,
        parquet_path,
        indices,
        pseudo_labels,
        tokenizer,
        row_group_starts,
        max_length=512
    ):
        self.indices = np.asarray(indices)
        self.pseudo_labels = np.asarray(pseudo_labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

        self.pf = pq.ParquetFile(parquet_path)
        self.row_group_starts = row_group_starts

        self.row_groups = np.searchsorted(
            row_group_starts,
            self.indices,
            side="right"
        ) - 1

        self.cache = {}

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        row_idx = self.indices[idx]
        group = self.row_groups[idx]
        local_idx = row_idx - self.row_group_starts[group]

        if group not in self.cache:
            table = self.pf.read_row_group(
                int(group),
                columns=["model_text"]
            )
            self.cache[group] = table["model_text"]

        text = self.cache[group][local_idx].as_py()

        if isinstance(text, bytes):
            text = text.decode(
                "utf-8",
                errors="replace"
            )

        encoded = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "label": torch.tensor(
                self.pseudo_labels[idx],
                dtype=torch.long
            )
        }

In [40]:
test_labels = np.zeros(
    len(sample_indices),
    dtype=np.int64
)

dataset_test = DeepClusterDataset(
    DATA_PATH,
    sample_indices,
    test_labels,
    tokenizer,
    row_group_starts,
    max_length=MAX_LENGTH
)

In [41]:
item = dataset_test[0]

print("input_ids:", item["input_ids"].shape)
print("attention_mask:", item["attention_mask"].shape)
print("label:", item["label"])

input_ids: torch.Size([512])
attention_mask: torch.Size([512])
label: tensor(0)


In [42]:
for idx in [0, 100, 1000, 10000, 499999]:
    item = dataset_test[idx]

    print(
        idx,
        item["input_ids"].shape,
        item["attention_mask"].sum().item()
    )

0 torch.Size([512]) 111
100 torch.Size([512]) 512
1000 torch.Size([512]) 171
10000 torch.Size([512]) 127
499999 torch.Size([512]) 512


Функция генерации embeddings

In [43]:
def generate_embeddings(
    texts,
    batch_size=16,
    max_length=512
):
    encoder.eval()

    all_embeddings = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        input_ids = encoded["input_ids"].to(DEVICE)
        attention_mask = encoded["attention_mask"].to(DEVICE)

        with torch.no_grad():
            output = encoder(
                input_ids,
                attention_mask
            )

        embeddings = output[:, 0, :]
        all_embeddings.append(
            embeddings.cpu().numpy()
        )

        if start % 10000 == 0:
            print(
                f"{start:,}/{len(texts):,}"
            )

    return np.concatenate(
        all_embeddings,
        axis=0
    )

In [44]:
class EmbeddingDataset(Dataset):
    def __init__(
        self,
        parquet_path,
        indices,
        tokenizer,
        row_group_starts,
        max_length=512
    ):
        self.indices = np.asarray(indices)
        self.tokenizer = tokenizer
        self.max_length = max_length

        self.pf = pq.ParquetFile(parquet_path)
        self.row_group_starts = row_group_starts

        self.row_groups = np.searchsorted(
            row_group_starts,
            self.indices,
            side="right"
        ) - 1

        self.cache = {}

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        row_idx = self.indices[idx]
        group = self.row_groups[idx]

        local_idx = (
            row_idx -
            self.row_group_starts[group]
        )

        if group not in self.cache:
            table = self.pf.read_row_group(
                int(group),
                columns=["model_text"]
            )
            self.cache[group] = table["model_text"]

        text = self.cache[group][local_idx].as_py()

        if isinstance(text, bytes):
            text = text.decode(
                "utf-8",
                errors="replace"
            )

        encoded = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0)
        }

In [45]:
embedding_dataset = EmbeddingDataset(
    DATA_PATH,
    sample_indices,
    tokenizer,
    row_group_starts,
    max_length=MAX_LENGTH
)

embedding_loader = DataLoader(
    embedding_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Dataset size:", len(embedding_dataset))

Dataset size: 500000


In [46]:
MAX_LENGTH = 128
BATCH_SIZE = 32

print("Max length:", MAX_LENGTH)
print("Batch size:", BATCH_SIZE)

Max length: 128
Batch size: 32


In [47]:
embedding_dataset = EmbeddingDataset(
    DATA_PATH,
    sample_indices,
    tokenizer,
    row_group_starts,
    max_length=MAX_LENGTH
)

embedding_loader = DataLoader(
    embedding_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Dataset size:", len(embedding_dataset))
print("Batches:", len(embedding_loader))

Dataset size: 500000
Batches: 15625


In [50]:
BATCH_SIZE = 16

TEST_SIZE = 100

In [53]:
TEST_SIZE = 100
BATCH_SIZE = 16

embedding_dataset_test = torch.utils.data.Subset(
    embedding_dataset,
    range(TEST_SIZE)
)

embedding_loader = DataLoader(
    embedding_dataset_test,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Dataset size:", len(embedding_loader.dataset))
print("Batches:", len(embedding_loader))

Dataset size: 100
Batches: 7


In [54]:
encoder.eval()

embeddings = []

total_batches = len(embedding_loader)
total_samples = len(embedding_loader.dataset)

for batch_idx, batch in enumerate(embedding_loader, start=1):

    input_ids = batch["input_ids"].to(DEVICE)
    attention_mask = batch["attention_mask"].to(DEVICE)

    with torch.no_grad():
        output = encoder(
            input_ids,
            attention_mask
        )

    embeddings.append(
        output[:, 0, :].cpu().numpy()
    )

    processed = min(
        batch_idx * BATCH_SIZE,
        total_samples
    )

    print(
        f"\rБатч: {batch_idx}/{total_batches} | "
        f"Обработано: {processed}/{total_samples} | "
        f"Прогресс: {processed / total_samples * 100:.1f}%",
        end="",
        flush=True
    )

X_deep = np.concatenate(embeddings, axis=0)

print()
print("Shape:", X_deep.shape)

Батч: 7/7 | Обработано: 100/100 | Прогресс: 100.0%
Shape: (100, 312)


In [56]:
original_embeddings_raw = []

for batch in parquet_file.iter_batches(
    batch_size=100,
    columns=["embedding"]
):
    original_embeddings_raw.extend(
        batch.column("embedding").to_pylist()
    )

    if len(original_embeddings_raw) >= 100:
        break

original_embeddings = np.array([
    decode_embedding(x)
    for x in original_embeddings_raw[:100]
])

print("Original:", original_embeddings.shape)
print("New:", X_deep.shape)

Original: (100, 312)
New: (100, 312)


In [57]:
similarities = []

for i in range(len(X_deep)):
    sim = cosine_similarity(
        X_deep[i:i+1],
        original_embeddings[i:i+1]
    )[0, 0]

    similarities.append(sim)

print("Средняя cosine similarity:", np.mean(similarities))
print("Минимальная:", np.min(similarities))
print("Максимальная:", np.max(similarities))

Средняя cosine similarity: 0.044747863
Минимальная: -0.1436636
Максимальная: 0.30766368


In [58]:
text = texts_test[0]

encoded = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=128
)

input_ids = encoded["input_ids"].to(DEVICE)
attention_mask = encoded["attention_mask"].to(DEVICE)

with torch.no_grad():
    output = encoder(
        input_ids,
        attention_mask
    )

new_embedding = output[:, 0, :].cpu().numpy()[0]

original_embedding = decode_embedding(
    embeddings_test_raw[0]
)

similarity = cosine_similarity(
    new_embedding.reshape(1, -1),
    original_embedding.reshape(1, -1)
)[0, 0]

print("Text:", text[:200])
print("New embedding:", new_embedding.shape)
print("Original embedding:", original_embedding.shape)
print("Cosine similarity:", similarity)

Text: Дисплей для OPPO Reno 13 F 4G In-Cell ЧерныйНе определенДисплей для OPPO Reno 13 F 4G In-Cell Черный идеально подойдет для замены Вашего разбитого дисплея. Однако, обратите внимание на важные моменты:
New embedding: (312,)
Original embedding: (312,)
Cosine similarity: 0.9922245


In [59]:
texts_100 = []
original_embeddings_raw = []

for batch in parquet_file.iter_batches(
    batch_size=100,
    columns=["model_text", "embedding"]
):
    texts_100 = batch.column("model_text").to_pylist()
    original_embeddings_raw = batch.column("embedding").to_pylist()
    break

texts_100 = [
    x.decode("utf-8", errors="replace")
    if isinstance(x, bytes)
    else x
    for x in texts_100
]

original_embeddings = np.array([
    decode_embedding(x)
    for x in original_embeddings_raw
])

print("Texts:", len(texts_100))
print("Original embeddings:", original_embeddings.shape)

Texts: 100
Original embeddings: (100, 312)


In [60]:
from torch.utils.data import Dataset, DataLoader


class TextDataset(Dataset):
    def __init__(self, texts, tokenizer):
        self.texts = texts
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=128,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0)
        }


test_dataset = TextDataset(
    texts_100,
    tokenizer
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

print("Dataset:", len(test_dataset))
print("Batches:", len(test_loader))

Dataset: 100
Batches: 7


In [61]:
encoder.eval()

new_embeddings = []

total_batches = len(test_loader)
total_samples = len(test_dataset)

for batch_idx, batch in enumerate(test_loader, start=1):

    input_ids = batch["input_ids"].to(DEVICE)
    attention_mask = batch["attention_mask"].to(DEVICE)

    with torch.no_grad():
        output = encoder(
            input_ids,
            attention_mask
        )

    new_embeddings.append(
        output[:, 0, :].cpu().numpy()
    )

    processed = min(
        batch_idx * 16,
        total_samples
    )

    print(
        f"\rБатч: {batch_idx}/{total_batches} | "
        f"Обработано: {processed}/{total_samples} | "
        f"Прогресс: {processed / total_samples * 100:.1f}%",
        end="",
        flush=True
    )

X_deep = np.concatenate(
    new_embeddings,
    axis=0
)

print()
print("New embeddings:", X_deep.shape)

Батч: 7/7 | Обработано: 100/100 | Прогресс: 100.0%
New embeddings: (100, 312)


In [62]:
similarities = np.sum(
    X_deep * original_embeddings,
    axis=1
) / (
    np.linalg.norm(X_deep, axis=1)
    * np.linalg.norm(original_embeddings, axis=1)
)

print("Средняя cosine similarity:", similarities.mean())
print("Минимальная:", similarities.min())
print("Максимальная:", similarities.max())

Средняя cosine similarity: 0.9882128
Минимальная: 0.9279724
Максимальная: 0.9979233


In [64]:
SAMPLE_SIZE = 100_000

sample_indices = np.asarray(sample_indices)

texts = []

for batch in parquet_file.iter_batches(
    columns=["model_text"],
    batch_size=10_000
):
    batch_texts = batch.column("model_text").to_pylist()
    texts.extend(batch_texts)

    if len(texts) >= SAMPLE_SIZE:
        break

texts = np.array([
    x.decode("utf-8", errors="replace")
    if isinstance(x, bytes)
    else x
    for x in texts[:SAMPLE_SIZE]
], dtype=object)

print("Texts:", texts.shape)

Texts: (100000,)


In [65]:
BATCH_SIZE = 32

deep_dataset = TextDataset(
    texts,
    tokenizer
)

deep_loader = DataLoader(
    deep_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Dataset:", len(deep_dataset))
print("Batches:", len(deep_loader))

Dataset: 100000
Batches: 3125


In [66]:
encoder.eval()

deep_embeddings = []

total_batches = len(deep_loader)
total_samples = len(deep_dataset)

for batch_idx, batch in enumerate(deep_loader, start=1):

    input_ids = batch["input_ids"].to(DEVICE)
    attention_mask = batch["attention_mask"].to(DEVICE)

    with torch.no_grad():
        output = encoder(
            input_ids,
            attention_mask
        )

    deep_embeddings.append(
        output[:, 0, :].cpu().numpy()
    )

    processed = min(
        batch_idx * BATCH_SIZE,
        total_samples
    )

    print(
        f"\rБатч: {batch_idx}/{total_batches} | "
        f"Обработано: {processed}/{total_samples} | "
        f"Прогресс: {processed / total_samples * 100:.1f}%",
        end="",
        flush=True
    )

X_deep = np.concatenate(
    deep_embeddings,
    axis=0
)

print()
print("Shape:", X_deep.shape)

Батч: 3125/3125 | Обработано: 100000/100000 | Прогресс: 100.0%
Shape: (100000, 312)


In [67]:
print("Shape:", X_deep.shape)
print("NaN:", np.isnan(X_deep).sum())
print("Inf:", np.isinf(X_deep).sum())

norms = np.linalg.norm(X_deep, axis=1)

print("Norm min:", norms.min())
print("Norm mean:", norms.mean())
print("Norm max:", norms.max())

Shape: (100000, 312)
NaN: 0
Inf: 0
Norm min: 11.47379
Norm mean: 15.049506
Norm max: 16.614403


In [69]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

In [70]:
pca_deep = PCA(
    n_components=256,
    whiten=True,
    random_state=RANDOM_STATE
)

X_deep_pca = pca_deep.fit_transform(X_deep)

X_deep_cluster = normalize(
    X_deep_pca,
    norm="l2"
)

print("Before:", X_deep.shape)
print("After PCA:", X_deep_pca.shape)
print("After L2:", X_deep_cluster.shape)

Before: (100000, 312)
After PCA: (100000, 256)
After L2: (100000, 256)


In [71]:
from sklearn.cluster import KMeans

N_CLUSTERS = 10

kmeans_deep = KMeans(
    n_clusters=N_CLUSTERS,
    n_init=10,
    max_iter=300,
    random_state=RANDOM_STATE
)

labels_deep = kmeans_deep.fit_predict(
    X_deep_cluster
)

print("Labels:", labels_deep.shape)
print("Clusters:", np.unique(labels_deep))

Python(66197) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Labels: (100000,)
Clusters: [0 1 2 3 4 5 6 7 8 9]


In [73]:
table = parquet_file.read(
    columns=["category_id"]
)

selected = table.take(
    sample_indices
)

y = np.array(
    selected.column("category_id").to_pylist()
)

print("y shape:", y.shape)
print("Unique categories:", len(np.unique(y)))

y shape: (500000,)
Unique categories: 122


In [ ]:
table = parquet_file.read(
    columns=["model_text", "category_id"]
)

all_texts = table.column("model_text").to_pylist()
all_categories = table.column("category_id").to_pylist()

text_to_category = {}

for text, category in zip(all_texts, all_categories):
    if isinstance(text, bytes):
        text = text.decode("utf-8", errors="replace")
    text_to_category[text] = category

y_deep = np.array(
    [text_to_category[text] for text in texts],
    dtype=np.int64
)

print("y_deep:", y_deep.shape)
print("Unique categories:", len(np.unique(y_deep)))

: 

In [77]:
print("X_deep:", X_deep.shape)
print("labels_deep:", labels_deep.shape)
print("sample_indices:", sample_indices.shape)
print("sample_indices min:", sample_indices.min())
print("sample_indices max:", sample_indices.max())

X_deep: (100000, 312)
labels_deep: (100000,)
sample_indices: (500000,)
sample_indices min: 5
sample_indices max: 4793818


In [78]:
print("texts:", texts.shape)
print("First text:", texts[0])
print("Last text:", texts[-1])

texts: (100000,)
First text: Дисплей для OPPO Reno 13 F 4G In-Cell ЧерныйНе определенДисплей для OPPO Reno 13 F 4G In-Cell Черный идеально подойдет для замены Вашего разбитого дисплея. Однако, обратите внимание на важные моменты: <br /> 1. Не забудьте выключить телефон перед заменой дисплейного модуля.<br /> 2. Проведите предварительную проверку дисплея, просто подключив его к телефону, убедитесь, что всё корректно работает, и только после этого приступайте к установке. Этот пункт крайне важен так как: гарантия НЕ распространяется на дисплеи после установки.<br /> 3. Если на задней стороне дисплея присутствует двухсторонний скотч для фиксации шлейфа, обязательно приклейте шлейф с его помощью, в противном случае тачскрин будет работать некорректно.<br /> 4. Не прилагайте усилий при установке, при правильной установке, дисплей помещается в посадочное место без усилий.<br /> 5. При прокладке шлейфов ни в коем случае не допускайте заломов.<br /> 6. Если под старым дисплеем были какие-либо 